# Huấn luyện dự đoán nhu cầu

Thực hiện từng bước theo bài lab hồi quy và chuỗi thời gian. Dữ liệu nguồn 01/2022–07/2023; không đổi ngày hoặc bổ sung lượng bán giả.

In [1]:
from pathlib import Path
from datetime import datetime
import os
import sys
import pandas as pd

RESEARCH = Path.cwd()
if not (RESEARCH / "train.py").exists():
    RESEARCH = RESEARCH / "Ma_nguon/nghien_cuu"
sys.path.insert(0, str(RESEARCH.resolve()))
from train import load_data, train_models, evaluate, verify_reference, save_results, plot_product
from store.forecast.training import make_samples

DATA_DIR = Path(os.environ.get("DACN_DATA_DIR", RESEARCH.parents[1] / "Du_lieu"))
OUTPUT = RESEARCH / "ket_qua_chay_lai" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
PRODUCT = "MH00381"


## 1. Đọc và kiểm tra dữ liệu

Tháng chưa quan sát giữ trống, không coi là lượng bán bằng 0.

In [2]:
data = load_data(DATA_DIR)
print("Số mã nguồn:", len(data["products"]))
print("Số tháng:", len(data["months"]))
print("Số dòng quan sát:", len(data["rows"]))
print("Mã băm dữ liệu:", data["source_sha256"])


Số mã nguồn: 30366
Số tháng: 19
Số dòng quan sát: 188537
Mã băm dữ liệu: 11c9481ff0fc8558bbb11b1d5f7117345db9ac49a522f81052d963c292bb86af


## 2. Tạo mẫu học từ quá khứ

Ví dụ mốc 01/2023 chỉ dùng lịch sử đến 12/2022. X gồm ba lượng bán trễ, trung bình ba tháng và tháng cần dự đoán; y là lượng bán của tháng đích trong tập học.

In [3]:
X_train, y_train, X_next, eligible_ids = make_samples(data["matrix"][:, :12], data["months"][:12])
sample = pd.DataFrame(X_train[:5], columns=["lag1", "lag2", "lag3", "ma3", "month_number"])
sample["target_qty"] = y_train[:5]
print(sample.to_string(index=False))
print("Mẫu học:", X_train.shape, "Mã đủ điều kiện:", len(eligible_ids))


 lag1  lag2  lag3       ma3  month_number  target_qty
 11.0  19.0  43.0 24.333333           4.0        58.0
  7.0   1.0  22.0 10.000000           4.0        11.0
 21.0  15.0   4.0 13.333333           4.0        29.0
  8.0  18.0  26.0 17.333333           4.0        10.0
 18.0  11.0  23.0 17.333333           4.0        27.0
Mẫu học: (22461, 5) Mã đủ điều kiện: 1844


## 3. Huấn luyện theo từng kỳ

Cấu hình chốt trên Validation 01–03/2023. Test 04–07/2023; dự đoán tiếp 08/2023. Mỗi kỳ chỉ nhận lịch sử trước tháng đích. Website và notebook dùng cùng mã tính toán và Pipeline.

In [4]:
predictions, origins = train_models(data)
selected, metrics = evaluate(data, predictions)
print("Phương pháp được chọn:", selected)
print(pd.DataFrame(origins).to_string(index=False))


01/2023: 22,461 mẫu học, 1,844 mã dự đoán, 1,582 cặp có thực tế.
02/2023: 26,257 mẫu học, 2,061 mã dự đoán, 1,431 cặp có thực tế.
03/2023: 29,932 mẫu học, 1,865 mã dự đoán, 1,548 cặp có thực tế.
04/2023: 35,422 mẫu học, 2,277 mã dự đoán, 1,989 cặp có thực tế.
05/2023: 39,055 mẫu học, 2,320 mã dự đoán, 1,977 cặp có thực tế.
06/2023: 41,536 mẫu học, 2,255 mã dự đoán, 1,913 cặp có thực tế.
07/2023: 45,237 mẫu học, 2,366 mã dự đoán, 2,076 cặp có thực tế.
08/2023: 51,651 mẫu học, 2,929 mã dự đoán, 0 cặp có thực tế.
Phương pháp được chọn: random_forest
    target     cutoff  training_pairs  candidate_skus  scored_pairs
2023-01-01 2022-12-01           22461            1844          1582
2023-02-01 2023-01-01           26257            2061          1431
2023-03-01 2023-02-01           29932            1865          1548
2023-04-01 2023-03-01           35422            2277          1989
2023-05-01 2023-04-01           39055            2320          1977
2023-06-01 2023-05-01           41536  

## 4. Đánh giá và đối chiếu

Bốn phương pháp dùng cùng các cặp có thực tế. Không dùng Accuracy phân loại hoặc diễn giải 100 − WAPE thành độ chính xác. Bộ Test đã được xem trước đây; đây là chạy lại kết quả.

In [5]:
scores = pd.DataFrame(metrics["test"]).T
print(scores[["n", "mae", "rmse", "wape_pct", "mase", "bias"]].to_string())
checked = verify_reference(data, predictions, metrics, DATA_DIR)
print("Số kết quả đã khớp:", checked)


                    n       mae       rmse   wape_pct      mase      bias
naive          7955.0  5.651288  16.811821  65.870562  0.553966 -0.182527
ma3            7955.0  5.161576  13.691076  60.162542  0.513949  0.370543
linear         7955.0  4.794603  14.835507  55.885168  0.476589 -2.484244
random_forest  7955.0  4.790775  14.027800  55.840550  0.487334 -1.987531
Số kết quả đã khớp: {'test_pairs': 7955, 'future_skus': 2929}


## 5. Dự đoán và thực tế theo sản phẩm

Chênh lệch = Dự đoán − Thực tế. Kỳ chưa có thực tế giữ trống. Đơn vị theo dữ liệu nguồn, không tự quy đổi thành số đôi.

In [6]:
product_rows = pd.DataFrame([r for r in predictions if r["unit_alias"] == PRODUCT and r["phase"] != "validation"])
product_rows["difference"] = product_rows.apply(lambda r: r[selected] - r["actual"] if r["actual"] != "" else None, axis=1)
print(product_rows[["unit_alias", "month", selected, "actual", "difference"]].to_string(index=False))
result = save_results(OUTPUT, data, predictions, origins, selected, metrics, checked, None)
plot_product(predictions, PRODUCT, selected, OUTPUT)
print("Đã lưu bảng, chỉ số và biểu đồ tại:", OUTPUT.relative_to(RESEARCH))


unit_alias   month  random_forest actual  difference
   MH00381 2023-04      35.217957   20.0   15.217957
   MH00381 2023-05      18.664285   27.0   -8.335715
   MH00381 2023-06      14.875865   23.0   -8.124135
   MH00381 2023-07      14.440625   25.0  -10.559375
   MH00381 2023-08      14.890428                NaN
Đã lưu bảng, chỉ số và biểu đồ tại: ket_qua_chay_lai\20261124_211610_155874


Mở ảnh `du_bao_<ma>.png` trong thư mục kết quả vừa in. Đổi PRODUCT sang mã đủ lịch sử để xem sản phẩm khác. Lần chạy mới không ghi đè kết quả đã công bố.